# Exp0.1 — Checkpoint firing-pattern diagnostic

Checkpoint-only diagnostic for `experiment_0_1_general_comparison`. **This notebook never trains, backpropagates, or updates weights.**

Fixed comparison:
- architectures: `short_mid`, `mid_long`, `short_mid_long`
- objectives: `whole_count_ce`, `timestep_ce`
- hidden communication: `binary` (`hidden_cap=1`) vs `multi_h` (`hidden_cap=31`)
- output layer remains binary (`output_cap=1`)
- same Exp0.1 split, same seed, same test segment for every checkpoint
- original full padded window is traced; the red dashed line marks `valid_length`, and the shaded tail is padding used only for dynamics diagnosis

For every hidden layer the notebook records the exact forward states

\[I_t,\qquad U_t^{-},\qquad U_t,\qquad S_t.\]

Binary hidden spikes are shown as rasters. Multi-H hidden event counts are shown as heatmaps so the integer event amplitude is preserved. Signed `I`, `U^-`, and post-reset `U` are shown as symmetric-log heatmaps.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_VERSION = 'exp01_fire_pattern_v1_20260910'
start = Path.cwd().resolve()
repo_root = next((p for p in (start, *start.parents) if (p / 'scripts').is_dir() and (p / 'notebooks').is_dir()), None)
if repo_root is None:
    raise RuntimeError(f'Could not locate writingRing repo root from cwd={start}')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from IPython.display import Markdown, display

from scripts import experiment_0_1_general_comparison as exp01

print('NOTEBOOK_VERSION:', NOTEBOOK_VERSION)
print('repo_root:', repo_root)
print('Exp0.1 import: OK')

## Configuration and frozen checkpoint loading

The default `SAMPLE_INDEX=4` intentionally reuses the example segment from `experiment_0_1_5_wholecount_firing_patterns.ipynb` so the two diagnostics are directly comparable. Change it manually only when another specific segment is desired. Missing Exp0.1 checkpoints are errors; there is no training fallback.

In [ ]:
SEED = 11
SAMPLE_INDEX = 4
DEVICE = 'cpu'
BATCH_SIZE = 128
SAVE_FIGURES = False

ARCHITECTURES = ('short_mid', 'mid_long', 'short_mid_long')
OBJECTIVES = ('whole_count_ce', 'timestep_ce')
VARIANTS = (('binary', 1), ('multi_h', 31))

if tuple(ARCHITECTURES) != ('short_mid', 'mid_long', 'short_mid_long'):
    raise ValueError('This diagnostic is fixed to the three Exp0.1 direct-SNN backbones')
if tuple(OBJECTIVES) != ('whole_count_ce', 'timestep_ce'):
    raise ValueError('This diagnostic is fixed to the two Exp0.1 direct objectives')

results_root = exp01.results_dir(repo_root)
config = exp01.Config(
    repo_root=repo_root,
    results_dir=results_root,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    threads=1,
)
device = torch.device(DEVICE)
data = exp01.prepare_data(repo_root)
X, y, lengths = data.Xte, data.yte, data.lte
FULL_T = int(X.shape[1])

if not 0 <= int(SAMPLE_INDEX) < len(X):
    raise IndexError(f'SAMPLE_INDEX={SAMPLE_INDEX} outside 0..{len(X)-1}')
sample_index = int(SAMPLE_INDEX)
valid_length = int(lengths[sample_index])
if not 0 < valid_length <= FULL_T:
    raise ValueError(f'Invalid valid_length={valid_length} for FULL_T={FULL_T}')

def make_spec(architecture: str, objective: str, variant: str, hidden_cap: int) -> exp01.RunSpec:
    return exp01.RunSpec(
        family=exp01.DIRECT_FAMILY,
        architecture=architecture,
        objective=objective,
        variant=variant,
        hidden_cap=int(hidden_cap),
        seed=SEED,
    )

selected = {}
for objective in OBJECTIVES:
    for architecture in ARCHITECTURES:
        for variant, hidden_cap in VARIANTS:
            spec = make_spec(architecture, objective, variant, hidden_cap)
            path = exp01.checkpoint_path(results_root, spec)
            if not path.exists():
                raise FileNotFoundError(f'Missing Exp0.1 checkpoint: {path}')
            model, checkpoint = exp01.load_model(spec, data, config)
            if not model.output_spiking or model.output_linear is None or model.output_lif is None:
                raise RuntimeError(f'{spec.key} is not the expected direct spiking-output model')
            selected[(objective, architecture, variant)] = {
                'spec': spec,
                'model': model,
                'checkpoint': checkpoint,
                'path': path,
            }

print(f'Loaded {len(selected)} / 12 existing Exp0.1 checkpoints')
print('seed:', SEED)
print('sample_index:', sample_index)
print('true_label:', data.labels[int(y[sample_index])])
print('valid_length:', valid_length, 'full_T:', FULL_T, 'sampling_rate_hz:', data.fs)
print('architectures:', {k: exp01.DIRECT_ARCHITECTURES[k] for k in ARCHITECTURES})

## Exact full-window state tracing

This reproduces the trained Exp0.1 forward equations exactly while recording hidden-layer synaptic state `I`, pre-reset membrane `U^-`, post-reset membrane `U`, and emitted events. The output layer is also traced for a strict spike-trajectory contract check, but the state heatmaps below focus on hidden layers where the multi-`tau_syn` dynamics live.

In [ ]:
def to_numpy(tensor: torch.Tensor) -> np.ndarray:
    return tensor.detach().cpu().numpy()

def trace_full_window(model: exp01.MultiTauHierarchySNN, padded_input: np.ndarray) -> dict[str, object]:
    x = torch.as_tensor(padded_input, dtype=torch.float32, device=device).unsqueeze(0)
    if int(x.shape[1]) != FULL_T:
        raise ValueError(f'Expected full padded T={FULL_T}, got {x.shape[1]}')

    batch = int(x.shape[0])
    syn_states = [
        torch.zeros(batch, exp01.HIDDEN_WIDTH, device=device, dtype=x.dtype)
        for _ in model.hidden_linears
    ]
    mem_states = [torch.zeros_like(value) for value in syn_states]
    output_mem = torch.zeros(batch, model.n_classes, device=device, dtype=x.dtype)

    layer_records = [
        {'I': [], 'U_pre': [], 'U_post': [], 'spikes': []}
        for _ in model.hidden_linears
    ]
    output_record = {'I': [], 'U_pre': [], 'U_post': [], 'spikes': []}

    with torch.inference_mode():
        for step in range(FULL_T):
            current = x[:, step]
            for index, (linear, lif) in enumerate(zip(model.hidden_linears, model.hidden_lifs, strict=True)):
                alpha = getattr(model, model._alpha_names[index])
                syn_states[index] = alpha * syn_states[index] + linear(current)
                spike, mem_states[index], pre_reset = lif(syn_states[index], mem_states[index])

                layer_records[index]['I'].append(syn_states[index].squeeze(0).clone())
                layer_records[index]['U_pre'].append(pre_reset.squeeze(0).clone())
                layer_records[index]['U_post'].append(mem_states[index].squeeze(0).clone())
                layer_records[index]['spikes'].append(spike.squeeze(0).clone())
                current = spike

            output_current = model.output_linear(current)
            output_spike, output_mem, output_pre = model.output_lif(output_current, output_mem)
            output_record['I'].append(output_current.squeeze(0).clone())
            output_record['U_pre'].append(output_pre.squeeze(0).clone())
            output_record['U_post'].append(output_mem.squeeze(0).clone())
            output_record['spikes'].append(output_spike.squeeze(0).clone())

    packed_layers = []
    for record in layer_records:
        packed_layers.append({key: to_numpy(torch.stack(values, dim=0)) for key, values in record.items()})
    packed_output = {key: to_numpy(torch.stack(values, dim=0)) for key, values in output_record.items()}
    return {'layers': tuple(packed_layers), 'output': packed_output}

traces = {}
for key, item in selected.items():
    traces[key] = trace_full_window(item['model'], X[sample_index])

# Strict contract check: the diagnostic trace must reproduce Exp0.1's native spike trajectory exactly.
xb = torch.as_tensor(X[sample_index:sample_index+1], dtype=torch.float32, device=device)
with torch.inference_mode():
    for key, item in selected.items():
        native = item['model'].forward_trajectory(xb)
        native_hidden = native['hidden_spikes']
        native_output = native['output_spikes']
        if not isinstance(native_hidden, tuple) or not isinstance(native_output, torch.Tensor):
            raise TypeError(f'Unexpected native trajectory payload for {item["spec"].key}')
        for layer_index, native_spikes in enumerate(native_hidden):
            expected = torch.as_tensor(traces[key]['layers'][layer_index]['spikes'], device=device)
            torch.testing.assert_close(expected, native_spikes[0], rtol=0.0, atol=0.0)
        expected_output = torch.as_tensor(traces[key]['output']['spikes'], device=device)
        torch.testing.assert_close(expected_output, native_output[0], rtol=0.0, atol=0.0)

print('Full-window trace contract check: PASS for all 12 checkpoints')

## Plot helpers

Hidden neurons are ordered by the exact shift groups used by the Exp0.1 alpha vectors. The red dashed line is the valid endpoint; the shaded region is padding. For `I`, `U^-`, and `U`, each checkpoint uses one symmetric-log scale shared across all of its hidden layers.

In [ ]:
def group_slices(shifts: tuple[int, ...], width: int = exp01.HIDDEN_WIDTH) -> dict[int, slice]:
    shifts = tuple(int(s) for s in shifts)
    base, remainder = divmod(int(width), len(shifts))
    groups = {}
    start_idx = 0
    for index, shift in enumerate(shifts):
        size = base + (1 if index < remainder else 0)
        groups[shift] = slice(start_idx, start_idx + size)
        start_idx += size
    if start_idx != width:
        raise RuntimeError(f'Shift grouping covers {start_idx} neurons, expected {width}')
    return groups

def mark_valid(ax) -> None:
    boundary = valid_length - 0.5
    ax.axvline(boundary, color='red', linestyle='--', linewidth=1.4)
    if valid_length < FULL_T:
        ax.axvspan(boundary, FULL_T - 0.5, color='0.9', alpha=0.45)
    ax.set_xlim(-0.5, FULL_T - 0.5)

def decorate_hidden_axis(ax, shifts: tuple[int, ...]) -> None:
    centers, labels = [], []
    groups = group_slices(shifts)
    for group_index, (shift, group) in enumerate(groups.items()):
        centers.append((group.start + group.stop - 1) / 2.0)
        labels.append(f's{shift}')
        if group_index > 0:
            ax.axhline(group.start - 0.5, color='0.35', linewidth=0.6)
    ax.set_yticks(centers)
    ax.set_yticklabels(labels)
    ax.set_ylim(exp01.HIDDEN_WIDTH - 0.5, -0.5)

def state_norm(trace: dict[str, object], state_key: str) -> SymLogNorm:
    arrays = [np.asarray(layer[state_key]) for layer in trace['layers']]
    vmax = max(float(np.max(np.abs(array))) for array in arrays)
    threshold = float(exp01.THRESHOLD)
    vmax = max(vmax, threshold)
    return SymLogNorm(linthresh=threshold, vmin=-vmax, vmax=vmax, base=10)

def figure_output_dir(objective: str, architecture: str, variant: str) -> Path:
    return (
        results_root
        / 'fire_pattern_diagnostic'
        / f'seed{SEED}'
        / f'test_sample{sample_index:04d}'
        / objective
        / architecture
        / variant
    )

def maybe_save(fig, objective: str, architecture: str, variant: str, filename: str) -> None:
    if not SAVE_FIGURES:
        return
    out_dir = figure_output_dir(objective, architecture, variant)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / filename, dpi=180, bbox_inches='tight')

In [ ]:
def plot_firing_pattern(objective: str, architecture: str, variant: str):
    key = (objective, architecture, variant)
    trace = traces[key]
    spec = selected[key]['spec']
    layer_shifts = exp01.DIRECT_ARCHITECTURES[architecture]
    n_hidden = len(layer_shifts)
    fig, axes = plt.subplots(n_hidden + 1, 1, figsize=(16, 2.8 * (n_hidden + 1)), sharex=True)
    if n_hidden == 0:
        axes = np.asarray([axes])

    hidden_image = None
    for layer_index, shifts in enumerate(layer_shifts):
        spikes = np.asarray(trace['layers'][layer_index]['spikes'])
        ax = axes[layer_index]
        if variant == 'binary':
            ts, neurons = np.nonzero(spikes > 0.5)
            ax.scatter(ts, neurons, s=5, c='black', marker='.')
        elif variant == 'multi_h':
            hidden_image = ax.imshow(
                spikes.T,
                aspect='auto',
                interpolation='nearest',
                cmap='magma',
                vmin=0.0,
                vmax=float(spec.hidden_cap),
            )
        else:
            raise ValueError(f'Unknown variant: {variant}')
        decorate_hidden_axis(ax, shifts)
        mark_valid(ax)
        ax.set_ylabel(f'L{layer_index + 1}')

    output_spikes = np.asarray(trace['output']['spikes'])
    ts, neurons = np.nonzero(output_spikes > 0.5)
    axes[-1].scatter(ts, neurons, s=9, c='black', marker='.')
    axes[-1].set_yticks(range(len(data.labels)))
    axes[-1].set_yticklabels([str(label) for label in data.labels])
    axes[-1].set_ylim(len(data.labels) - 0.5, -0.5)
    axes[-1].set_ylabel('OUT')
    axes[-1].set_xlabel('Timestep')
    mark_valid(axes[-1])

    if hidden_image is not None:
        fig.colorbar(hidden_image, ax=list(axes[:n_hidden]), fraction=0.02, pad=0.015, label='Hidden events / timestep')
    fig.suptitle(
        f'{architecture} | {objective} | {variant} | hidden_cap={spec.hidden_cap} | '
        f'label={data.labels[int(y[sample_index])]} | valid={valid_length}/{FULL_T}'
    )
    fig.tight_layout(rect=(0, 0, 0.96, 0.95))
    maybe_save(fig, objective, architecture, variant, 'firing_pattern.png')
    return fig

def plot_hidden_state(objective: str, architecture: str, variant: str, state_key: str, title: str, colorbar_label: str):
    key = (objective, architecture, variant)
    trace = traces[key]
    layer_shifts = exp01.DIRECT_ARCHITECTURES[architecture]
    n_hidden = len(layer_shifts)
    norm = state_norm(trace, state_key)
    fig, axes = plt.subplots(n_hidden, 1, figsize=(16, 2.8 * n_hidden), sharex=True, squeeze=False)
    axes = axes[:, 0]
    image = None

    for layer_index, shifts in enumerate(layer_shifts):
        values = np.asarray(trace['layers'][layer_index][state_key])
        image = axes[layer_index].imshow(
            values.T,
            aspect='auto',
            interpolation='nearest',
            cmap='coolwarm',
            norm=norm,
        )
        decorate_hidden_axis(axes[layer_index], shifts)
        mark_valid(axes[layer_index])
        axes[layer_index].set_ylabel(f'L{layer_index + 1}')

    axes[-1].set_xlabel('Timestep')
    if image is not None:
        fig.colorbar(image, ax=list(axes), fraction=0.02, pad=0.015, label=colorbar_label)
    fig.suptitle(
        f'{title} | {architecture} | {objective} | {variant} | '
        f'label={data.labels[int(y[sample_index])]} | valid={valid_length}/{FULL_T}'
    )
    fig.tight_layout(rect=(0, 0, 0.96, 0.95))
    maybe_save(fig, objective, architecture, variant, f'{state_key}.png')
    return fig

def show_condition(objective: str, architecture: str, variant: str) -> None:
    display(Markdown(f'### `{architecture}` — `{objective}` — `{variant}`'))
    plot_firing_pattern(objective, architecture, variant)
    plt.show()
    plot_hidden_state(objective, architecture, variant, 'I', 'Hidden synaptic state $I_t$', 'Synaptic state I')
    plt.show()
    plot_hidden_state(objective, architecture, variant, 'U_pre', 'Hidden pre-reset membrane $U_t^{-}$', 'Pre-reset membrane U-')
    plt.show()
    plot_hidden_state(objective, architecture, variant, 'U_post', 'Hidden post-reset membrane $U_t$', 'Post-reset membrane U')
    plt.show()

## WholeCount-CE checkpoints

For each backbone, binary is shown first and multi-H second so the reset/backlog effect can be compared directly.

In [ ]:
for architecture in ARCHITECTURES:
    for variant, _ in VARIANTS:
        show_condition('whole_count_ce', architecture, variant)

## Timestep-CE checkpoints

The plotting contract is identical to the WholeCount section; only the frozen training objective changes.

In [ ]:
for architecture in ARCHITECTURES:
    for variant, _ in VARIANTS:
        show_condition('timestep_ce', architecture, variant)

## Interpretation guide

Read each condition causally as

\[I_t \rightarrow U_t^{-} \rightarrow S_t \rightarrow U_t \rightarrow I_{t+1}.\]

The key comparisons are: (1) whether long shift groups (`s6/s7`) retain large `I_t`; (2) whether this repeatedly drives `U_t^-` above threshold; (3) whether binary subtractive reset leaves a supra-threshold post-reset `U_t` backlog; and (4) whether multi-H removes that membrane backlog while persistent `I_t` still recreates excitation on the next timestep. Padding activity is diagnostic only and is not part of the original valid-window training objective.